# 04 — Impact Analysis

This notebook applies the saved baseline model to each drift batch and measures how performance degrades.  
We then use SHAP to identify which drifted features are responsible for the degradation.

**Contents**
1. Setup & load model
2. Evaluate model on each drift batch
3. Performance degradation over batches
4. Confusion matrix comparison
5. SHAP — root cause analysis
6. Summary

## 1. Setup & load model

In [ ]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score,
    recall_score, RocCurveDisplay
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

os.chdir(r'C:\Users\baxjo\OneDrive\Documenten\GitHub\datadrift-challenge')
os.makedirs('figures', exist_ok=True)
os.makedirs('reports', exist_ok=True)

In [ ]:
# Load saved model and supporting files
model         = joblib.load('models/baseline_model.pkl')
FEATURES      = joblib.load('models/feature_list.pkl')
amount_scaler = joblib.load('models/amount_scaler.pkl')
baseline_metrics = pd.read_csv('models/baseline_metrics.csv')

print('Model loaded:', type(model).__name__)
print('Features:', FEATURES)
print()
print('Baseline metrics (test set):')
print(baseline_metrics.to_string(index=False))

In [ ]:
# Load drift batches
batches = {
    'batch_1': pd.read_csv('data/drift_1.csv'),
    'batch_2': pd.read_csv('data/drift_2.csv'),
    'batch_3': pd.read_csv('data/drift_3.csv'),
    'batch_4': pd.read_csv('data/drift_4.csv'),
    'batch_5': pd.read_csv('data/drift_5.csv'),
}

def preprocess_batch(df, amount_scaler):
    """Apply same preprocessing as training: scale Amount and Time."""
    df = df.copy()
    time_scaler = StandardScaler()
    df['Amount_scaled'] = amount_scaler.transform(df[['Amount']])
    df['Time_scaled']   = time_scaler.fit_transform(df[['Time']])
    return df

for name, df in batches.items():
    print(f'{name}: {df.shape}, fraud={df.Class.sum()}, fraud_rate={df.Class.mean():.4%}')

## 2. Evaluate model on each drift batch

In [ ]:
results = []

# Add baseline as first row
results.append({
    'batch': 'baseline (test)',
    'auc_roc':   baseline_metrics['auc_roc'].values[0],
    'f1':        baseline_metrics['f1'].values[0],
    'precision': baseline_metrics['precision'].values[0],
    'recall':    baseline_metrics['recall'].values[0],
    'fraud_count': None,
    'fraud_rate_%': 0.1727
})

# Evaluate on each batch
for batch_name, batch_df in batches.items():
    df = preprocess_batch(batch_df, amount_scaler)
    X  = df[FEATURES]
    y  = df['Class']

    y_pred       = model.predict(X)
    y_pred_proba = model.predict_proba(X)[:, 1]

    results.append({
        'batch':       batch_name,
        'auc_roc':     round(roc_auc_score(y, y_pred_proba), 4),
        'f1':          round(f1_score(y, y_pred), 4),
        'precision':   round(precision_score(y, y_pred, zero_division=0), 4),
        'recall':      round(recall_score(y, y_pred), 4),
        'fraud_count': int(y.sum()),
        'fraud_rate_%': round(y.mean() * 100, 4)
    })
    print(f'{batch_name}: AUC={results[-1]["auc_roc"]:.4f}  F1={results[-1]["f1"]:.4f}  '
          f'Precision={results[-1]["precision"]:.4f}  Recall={results[-1]["recall"]:.4f}')

results_df = pd.DataFrame(results)
results_df.to_csv('reports/impact_metrics.csv', index=False)
print()
print('Saved to reports/impact_metrics.csv')
results_df

## 3. Performance degradation over batches

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics   = ['auc_roc', 'f1', 'precision', 'recall']
titles    = ['AUC-ROC', 'F1 Score', 'Precision', 'Recall']
colors    = ['steelblue', 'tomato', '#6aab7a', '#b87eb8']
batch_labels = results_df['batch'].tolist()

for ax, metric, title, color in zip(axes.flatten(), metrics, titles, colors):
    values = results_df[metric].tolist()
    ax.plot(batch_labels, values, marker='o', color=color, linewidth=2, markersize=7)
    ax.axhline(values[0], color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Baseline')
    
    # Shade degradation area
    for i in range(1, len(values)):
        if values[i] < values[0]:
            ax.fill_between([batch_labels[i-1], batch_labels[i]],
                            [values[0], values[0]],
                            [values[i-1], values[i]],
                            alpha=0.08, color='red')
    
    for i, v in enumerate(values):
        ax.annotate(f'{v:.3f}', (batch_labels[i], v),
                    textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=8)
    
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=8)
    ax.set_ylim(max(0, min(values) - 0.1), min(1.05, max(values) + 0.1))

plt.suptitle('Model performance degradation across drift batches', fontsize=14)
plt.tight_layout()
plt.savefig('figures/impact_metrics_over_time.png', bbox_inches='tight')
plt.show()

In [ ]:
# Degradation delta table (vs baseline)
baseline_row = results_df[results_df['batch'] == 'baseline (test)'].iloc[0]
delta_rows = []

for _, row in results_df[results_df['batch'] != 'baseline (test)'].iterrows():
    delta_rows.append({
        'batch': row['batch'],
        'fraud_rate_%': row['fraud_rate_%'],
        'ΔAUC-ROC':   round(row['auc_roc']   - baseline_row['auc_roc'],   4),
        'ΔF1':        round(row['f1']        - baseline_row['f1'],        4),
        'ΔPrecision': round(row['precision'] - baseline_row['precision'], 4),
        'ΔRecall':    round(row['recall']    - baseline_row['recall'],    4),
    })

delta_df = pd.DataFrame(delta_rows)
print('Performance delta vs baseline (negative = degradation):')
delta_df

## 4. Confusion matrix comparison

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (batch_name, batch_df) in zip(axes, batches.items()):
    df = preprocess_batch(batch_df, amount_scaler)
    X  = df[FEATURES]
    y  = df['Class']
    y_pred = model.predict(X)

    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'],
                cbar=False)
    f1  = f1_score(y, y_pred)
    rec = recall_score(y, y_pred)
    ax.set_title(f'{batch_name}\nF1={f1:.3f} | Recall={rec:.3f}', fontsize=9)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion matrices — all drift batches', fontsize=13)
plt.tight_layout()
plt.savefig('figures/impact_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves — all batches on one plot
fig, ax = plt.subplots(figsize=(8, 6))
batch_colors = ['#e07b54', '#5b8db8', '#6aab7a', '#b87eb8', '#b8a84a']

for (batch_name, batch_df), color in zip(batches.items(), batch_colors):
    df = preprocess_batch(batch_df, amount_scaler)
    X  = df[FEATURES]
    y  = df['Class']
    y_pred_proba = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_pred_proba)
    RocCurveDisplay.from_predictions(y, y_pred_proba, ax=ax,
                                     name=f'{batch_name} (AUC={auc:.3f})',
                                     color=color)

ax.plot([0,1],[0,1],'k--', alpha=0.3)
ax.set_title('ROC curves — all drift batches')
plt.tight_layout()
plt.savefig('figures/impact_roc_curves.png', bbox_inches='tight')
plt.show()

## 5. SHAP — root cause analysis

SHAP values explain which features drive the model's predictions.  
By comparing SHAP values on the training set vs drift batches, we can see which features the model is relying on differently.

In [ ]:
# Load training data for SHAP reference
train = pd.read_csv('data/creditcard.csv')
train['Amount_scaled'] = amount_scaler.transform(train[['Amount']])
from sklearn.preprocessing import StandardScaler
time_scaler = StandardScaler()
train['Time_scaled'] = time_scaler.fit_transform(train[['Time']])

# Sample for speed — SHAP on full 284k rows is slow
train_sample = train[FEATURES].sample(1000, random_state=42)

# Build SHAP explainer
print('Building SHAP explainer (this may take ~30 seconds)...')
explainer = shap.TreeExplainer(model)
shap_train = explainer.shap_values(train_sample)

# For binary classification, shap_values returns list [class0, class1]
# We want class 1 (fraud)
if isinstance(shap_train, list):
    shap_train = shap_train[1]

print('Done.')

In [ ]:
# SHAP summary plot — training data
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_train, train_sample, show=False, plot_size=None)
plt.title('SHAP feature importance — training data', fontsize=12)
plt.tight_layout()
plt.savefig('figures/shap_train_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
def get_shap_values(explainer, X):
    """Always return SHAP values for class 1 (fraud), handling all output shapes."""
    sv = explainer.shap_values(X)
    if isinstance(sv, list):
        return sv[1]
    elif hasattr(sv, 'ndim') and sv.ndim == 3:
        return sv[:, :, 1]
    return sv

# Compare SHAP mean absolute values: train vs batch_3 vs batch_5 (most drifted)
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
datasets  = [
    ('Training', train_sample, shap_train),
    ('Batch 3',  preprocess_batch(batches['batch_3'], amount_scaler)[FEATURES].sample(500, random_state=42), None),
    ('Batch 5',  preprocess_batch(batches['batch_5'], amount_scaler)[FEATURES].sample(500, random_state=42), None),
]

# Compute SHAP for batch 3 and 5
for i in range(1, 3):
    sv = get_shap_values(explainer, datasets[i][1])
    datasets[i] = (datasets[i][0], datasets[i][1], sv)

for ax, (name, X_sample, shap_vals) in zip(axes, datasets):
    mean_abs = pd.Series(
        np.abs(shap_vals).mean(axis=0),
        index=FEATURES
    ).sort_values(ascending=True).tail(15)
    
    mean_abs.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
    ax.set_title(f'Mean |SHAP| — {name}', fontsize=11)
    ax.set_xlabel('Mean |SHAP value|')

plt.suptitle('SHAP importance comparison: Training vs Batch 3 vs Batch 5', fontsize=13)
plt.tight_layout()
plt.savefig('figures/shap_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# SHAP importance shift — which features changed most between train and batch_5?
shap_b5_sample = preprocess_batch(batches['batch_5'], amount_scaler)[FEATURES].sample(500, random_state=42)
shap_b5 = get_shap_values(explainer, shap_b5_sample)

train_importance = pd.Series(np.abs(shap_train).mean(axis=0), index=FEATURES)
b5_importance    = pd.Series(np.abs(shap_b5).mean(axis=0),    index=FEATURES)
importance_shift = (b5_importance - train_importance).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['tomato' if v > 0 else 'steelblue' for v in importance_shift.values]
importance_shift.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('SHAP importance shift: Batch 5 vs Training\n(red = model relies MORE on this feature in batch 5)', fontsize=11)
ax.set_xlabel('Change in mean |SHAP value|')
plt.tight_layout()
plt.savefig('figures/shap_importance_shift.png', bbox_inches='tight')
plt.show()

## 6. Summary

In [ ]:
# Load drift summary from notebook 03
drift_summary = pd.read_csv('reports/drift_summary.csv')

# Merge with performance metrics
perf = results_df[results_df['batch'] != 'baseline (test)'].copy()
final = perf.merge(drift_summary[['batch','mean_psi','label_drift','top_drifted_features']], on='batch')

print('=== FULL IMPACT SUMMARY ===')
print()
print(final[['batch','fraud_rate_%','auc_roc','f1','recall','label_drift','top_drifted_features']].to_string(index=False))

print()
print('=== KEY FINDINGS ===')
worst_f1  = perf.loc[perf['f1'].idxmin(),  'batch']
worst_rec = perf.loc[perf['recall'].idxmin(), 'batch']
best_auc  = perf.loc[perf['auc_roc'].idxmax(), 'batch']
print(f'Worst F1 score     : {worst_f1}  ({perf.loc[perf["f1"].idxmin(), "f1"]:.4f})')
print(f'Worst Recall       : {worst_rec} ({perf.loc[perf["recall"].idxmin(), "recall"]:.4f})')
print(f'Best AUC-ROC batch : {best_auc}  ({perf.loc[perf["auc_roc"].idxmax(), "auc_roc"]:.4f})')
print(f'Baseline F1        : {baseline_metrics["f1"].values[0]:.4f}')
print(f'Baseline AUC-ROC   : {baseline_metrics["auc_roc"].values[0]:.4f}')